# Energy Asset Data Intake

Prepare and review collaborator data for the mu-star fixed-asset electricity model.

This notebook is intentionally separate from the PyPSA-Earth baseline notebooks. It does not optimise capacity. Its outputs are stable processed asset tables used by the topology and interruption workflow.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from mu_star_energy.intake import prepare_collaborator_data
from mu_star_energy.paths import incoming_energy_dir, processed_energy_dir, repo_root

REPO_ROOT = repo_root()
INPUT_DIR = incoming_energy_dir() / "collaborator"
OUTPUT_DIR = processed_energy_dir() / "collaborator"

prepared = prepare_collaborator_data(INPUT_DIR, OUTPUT_DIR)
prepared

## Processed Inventory

The generation register is a location/provenance template. Capacities, fuels and generator-to-substation assignments must be validated against CEB data before dispatch simulation.

In [ ]:
substations = gpd.read_parquet(prepared.substations)
routes = gpd.read_parquet(prepared.transmission_routes)
generation_points = gpd.read_parquet(prepared.generation_points)
generation_areas = gpd.read_parquet(prepared.generation_areas)
generation_register = pd.read_csv(prepared.generation_register_template)
monthly_peak = pd.read_csv(prepared.monthly_peak_demand, index_col="year")
annual_demand = pd.read_csv(prepared.annual_sector_demand)

inventory = pd.Series({
    "substations": len(substations),
    "transmission route features": len(routes),
    "generation point features": len(generation_points),
    "generation polygon features": len(generation_areas),
    "named generation register rows": len(generation_register),
    "register capacities populated": int(generation_register["capacity_mw"].notna().sum()),
    "register bus assignments populated": int(generation_register["connected_bus_id"].notna().sum()),
})
display(inventory.to_frame("count"))
display(generation_register[["asset_id", "name", "asset_type", "capacity_mw", "connected_bus_id", "status"]])

## Asset Map

Generation footprints are plotted translucently above transmission routes. Named generation sites are shown as markers because many source polygons are too small to see at island scale.

In [ ]:
category_colors = {
    "thermal": "#8e24aa",
    "hydro": "#16a34a",
    "solar": "#c026d3",
    "wind": "#2563eb",
    "substation": "#f97316",
    "unspecified": "#9ca3af",
}
category_markers = {"thermal": "s", "hydro": "^", "solar": "v", "wind": "x", "unspecified": "D"}

fig, ax = plt.subplots(figsize=(10, 10))
routes.plot(ax=ax, color="#dc2626", linewidth=1.3, alpha=0.75, zorder=2)
for category, group in generation_areas.groupby("category"):
    alpha = 0.18 if category == "unspecified" else 0.4
    group.plot(
        ax=ax,
        color=category_colors.get(category, "#777777"),
        edgecolor="black" if category != "unspecified" else "none",
        linewidth=0.3,
        alpha=alpha,
        zorder=4,
    )
substations.plot(ax=ax, color="#ff8c00", edgecolor="black", markersize=42, zorder=6)
for category, group in generation_register.groupby("asset_type"):
    ax.scatter(
        group["lon"],
        group["lat"],
        c=category_colors.get(category, "#777777"),
        marker=category_markers.get(category, "D"),
        s=95,
        edgecolors="black" if category != "wind" else None,
        linewidths=1,
        label=category,
        zorder=8,
    )
ax.set_title("Collaborator transmission, substations and generation sites")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend(title="Named generation", fontsize="small")
plt.show()

## Demand Evidence

The workbook supplies observed monthly system peaks and annual sector totals. It does not yet supply the nodal hourly demand required by the operational model.

In [ ]:
display(monthly_peak.tail(8).style.format("{:.1f}"))
display(
    annual_demand.pivot(index="year", columns="category", values="demand_gwh")
    .tail(8)
    .style.format("{:.1f}")
)

## Interpretation Rules

- Preserve raw source files unchanged under `data/incoming`.
- Do not infer plant capacity from polygon area.
- Do not treat unnamed generation polygons as confirmed power stations.
- Record manual CEB-map/report interpretation in a processed register with provenance.
- Clear notebook outputs before committing because the underlying data are private.